In [ ]:
# imports
import pandas as pd
import numpy as np

In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
# load data
DATA_PATH = library_path.parent / "data"

data_df = pd.read_csv(f"{DATA_PATH}/Excel table survival analysis.csv", sep="\t", encoding="utf-8")
data_df.head()

In [ ]:
# selecting columns to use for analysis
cols_to_use = [
    "Bday", "OPDate", "Sex", "Age", "Tumor","sPCI", "pPCI", "Tod Datum", "Datum Rezidiv", "CC"
]

reduc_df = data_df[cols_to_use].copy()
reduc_df.head()

In [ ]:
# converting date columns to datetime format
reduc_df["Bday"] = pd.to_datetime(reduc_df["Bday"], errors="coerce")
reduc_df["OPDate"] = pd.to_datetime(reduc_df["OPDate"], errors="coerce")
reduc_df["Tod Datum"] = pd.to_datetime(reduc_df["Tod Datum"], errors="coerce")
reduc_df["Datum Rezidiv"] = pd.to_datetime(reduc_df["Datum Rezidiv"], errors="coerce")
reduc_df.head()

In [ ]:
# Time variable
reduc_df["time"] = np.where(
    reduc_df["Tod Datum"].notna(),
    (reduc_df["Tod Datum"] - reduc_df["OPDate"]).dt.days,
    np.where(
        reduc_df["Datum Rezidiv"].notna(),
        (reduc_df["Datum Rezidiv"] - reduc_df["OPDate"]).dt.days,
        np.nan
    )
)
reduc_df['months'] = reduc_df['time'] / 30.44  # Convert days to months

# Event indicator
reduc_df["event"] = np.where(reduc_df["Tod Datum"].notna(), 1, 0)

In [ ]:
reduc_df["Age_calc"] = (reduc_df["OPDate"] - reduc_df["Bday"]).dt.days / 365.25

reduc_df["Age_final"] = reduc_df["Age"]
reduc_df.loc[reduc_df["Age_final"].isna(), "Age_final"] = reduc_df["Age_calc"]

reduc_df["Sex"] = reduc_df["Sex"]-1

In [ ]:
reduc_df.info()

In [ ]:
# there are very rare tumor types, we will drop the tumor types with less than 10 samples
tumor_counts = reduc_df["Tumor"].value_counts()
tumors_to_keep = tumor_counts[tumor_counts >= 10].index
reduc_df = reduc_df[reduc_df["Tumor"].isin(tumors_to_keep)].copy()
reduc_df["Tumor"].value_counts()

In [ ]:
reduc_df = reduc_df[(reduc_df['CC']==0) | (reduc_df['CC']==1)].reset_index(drop=True)  # keep only CC0 and CC1
reduc_df["CC"].value_counts()

In [ ]:
tumor_dummies = pd.get_dummies(reduc_df, columns=["Tumor"], drop_first=True)
tumor_dummies.columns
reduc_df = pd.concat([reduc_df["Tumor"], tumor_dummies], axis=1)
reduc_df.head()

In [ ]:
reduc_df.to_csv(f"{DATA_PATH}/GPT_processed_survival_data.csv", index=False)

In [ ]:
reduc_df.shape

In [ ]:
reduc_df.columns